<a href="https://colab.research.google.com/github/Crissizt/Estadistica_G1_2026_1/blob/main/Copia_de_5_Analisis_una_variable_Python_R_fitdistrplus_TAREA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de una variable: probabilidad frecuentista y ajuste de distribuciones en Python y R

**Curso:** Estadística Aplicada con Python y R  
**Tema:** Probabilidad frecuentista, tabla de distribución de frecuencias, visualización univariada y ajuste de datos experimentales a distribuciones teóricas.  
**Herramientas:** Python (`fitter`, `scipy`, `pandas`, `seaborn`) y R (`fitdistrplus`, `carData`).

---

## Propósito del notebook

Este notebook mantiene la demostración inicial de probabilidad frecuentista y amplía el cierre del tema con una comparación metodológica entre:

1. **Python con `fitter`**, útil para probar varias distribuciones automáticamente.
2. **R con `fitdistrplus`**, útil para ajustar distribuciones paramétricas, comparar criterios de bondad de ajuste y generar gráficos diagnósticos.

La actividad final propone ajustar dos variables:

- **Ingeniería Agrícola:** `Conduc` del dataset `Soils` del paquete `carData`, relacionada con la conductividad del suelo.
- **Ingeniería Agroindustrial:** `consumo` del archivo `acero.csv`, relacionada con el consumo del proceso industrial.

> **Criterio docente:** la mejor distribución no se selecciona solo por un número. Deben compararse histogramas, densidades, AIC, BIC, estadísticos de bondad de ajuste, plausibilidad técnica e interpretación del fenómeno.

In [ ]:
# ============================================================
# Instalación de paquetes requeridos en Google Colab
# ============================================================

# Ejecutar esta celda en Google Colab.
# En algunos entornos, puede ser necesario reiniciar el kernel
# después de instalar paquetes.

!pip install fitter

In [ ]:
# ============================================================
# Librerías de Python
# ============================================================

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats
from fitter import Fitter
from statsmodels.datasets import get_rdataset

# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# Semilla para reproducibilidad
np.random.seed(123)

---

# 1. Demostración de la probabilidad frecuentista

La probabilidad frecuentista interpreta la probabilidad como el límite al cual tiende la frecuencia relativa cuando el número de repeticiones del experimento aumenta.

Para una unidad de aire acondicionado `Ai`, la frecuencia relativa se calcula como:

\[
h_i = \frac{f_i}{N}
\]

donde:

- \(f_i\): frecuencia absoluta de la unidad `Ai`.
- \(N\): número total de simulaciones.
- \(h_i\): frecuencia relativa observada.

Si las seis unidades son igualmente probables, la probabilidad teórica de cada una es:

\[
P(A_i)=\frac{1}{6}\approx 0.1667
\]

In [ ]:
# ============================================================
# Simulación: selección aleatoria de una de 6 unidades de aire
# ============================================================

unidades = ["A1", "A2", "A3", "A4", "A5", "A6"]
n_simulaciones = [10, 100, 1_000, 10_000]

print("Convergencia de la frecuencia relativa")
print("Probabilidad teórica de cada unidad: 1/6 = 0.1667\n")

for n in n_simulaciones:
    muestra = pd.Series(np.random.choice(unidades, size=n, replace=True))
    frec_relativa = muestra.value_counts(normalize=True).sort_index()

    print(f"Resultados con N = {n}")
    print(frec_relativa.round(4).to_string())
    print("-" * 50)

---

# 2. Tabla de distribución de frecuencias para una variable continua

Para una variable cuantitativa continua, los datos pueden agruparse en intervalos.  
Una regla básica para definir el número de intervalos es la **regla de Sturges**:

\[
k = 1 + 3.322\log_{10}(n)
\]

donde \(k\) es el número aproximado de intervalos y \(n\) el tamaño de la muestra.

In [ ]:
def generar_tdf(datos, nombre_var="Variable"):
    """
    Genera una tabla de distribución de frecuencias para una variable cuantitativa.

    Parámetros
    ----------
    datos : array-like
        Vector con los datos numéricos.
    nombre_var : str
        Nombre de la variable analizada.

    Retorna
    -------
    pandas.DataFrame
        Tabla con límites de clase, marca de clase, frecuencia absoluta,
        frecuencia relativa, frecuencia absoluta acumulada y frecuencia relativa acumulada.
    """
    x = pd.Series(datos).dropna()
    x = x[np.isfinite(x)]

    n = len(x)
    k = int(np.ceil(1 + 3.322 * np.log10(n)))

    counts, edges = np.histogram(x, bins=k)

    tdf = pd.DataFrame({
        "Variable": nombre_var,
        "Linf": edges[:-1],
        "Lsup": edges[1:],
        "mc": (edges[:-1] + edges[1:]) / 2,
        "f": counts
    })

    tdf["h"] = tdf["f"] / n
    tdf["F"] = tdf["f"].cumsum()
    tdf["H"] = tdf["h"].cumsum()

    return tdf


# Ejemplo con el dataset California Housing disponible en scikit-learn
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()
df_housing = pd.DataFrame(housing.data, columns=housing.feature_names)

tdf_medinc = generar_tdf(df_housing["MedInc"], nombre_var="MedInc")
tdf_medinc

---

# 3. Visualización univariada: histograma y densidad KDE

El histograma permite observar la forma general de la distribución.  
La curva KDE es una estimación suavizada de la densidad empírica.

In [ ]:
# ============================================================
# Histograma y KDE
# ============================================================

data = df_housing["MedInc"].dropna()

sns.histplot(data, kde=True, stat="density", bins=30)
plt.title("Distribución de MedInc con histograma y KDE")
plt.xlabel("MedInc")
plt.ylabel("Densidad")
plt.show()

---

# 4. Ajuste visual con distribuciones teóricas en Python

Antes de automatizar el ajuste, conviene comparar visualmente algunas distribuciones candidatas.  
En este ejemplo se contrastan una distribución normal y una distribución gamma.

In [ ]:
# ============================================================
# Ajuste visual usando scipy.stats
# ============================================================

x = df_housing["MedInc"].dropna().to_numpy()

# Ajuste de parámetros
params_norm = stats.norm.fit(x)
params_gamma = stats.gamma.fit(x)

# Rango para graficar las densidades ajustadas
x_grid = np.linspace(x.min(), x.max(), 500)

pdf_norm = stats.norm.pdf(x_grid, *params_norm)
pdf_gamma = stats.gamma.pdf(x_grid, *params_gamma)

# Gráfico
sns.histplot(x, bins=30, stat="density", alpha=0.35, label="Datos")
plt.plot(x_grid, pdf_norm, label="Normal ajustada")
plt.plot(x_grid, pdf_gamma, label="Gamma ajustada")
plt.title("Comparación visual de distribuciones ajustadas")
plt.xlabel("MedInc")
plt.ylabel("Densidad")
plt.legend()
plt.show()

# Prueba Kolmogorov-Smirnov orientativa
# Nota: cuando los parámetros se estiman con los mismos datos,
# el p-valor debe interpretarse con prudencia.
ks_norm = stats.kstest(x, "norm", args=params_norm)
ks_gamma = stats.kstest(x, "gamma", args=params_gamma)

print("KS Normal:", ks_norm)
print("KS Gamma :", ks_gamma)

---

# 5. Selección automática con `fitter` en Python

La librería `fitter` permite ajustar varias distribuciones de `scipy.stats` y compararlas mediante criterios como:

- `sumsquare_error`
- `aic`
- `bic`
- `ks_statistic`
- `ks_pvalue`

> **Recomendación:** para datos positivos continuos se pueden probar inicialmente: `norm`, `gamma`, `lognorm`, `expon` y `weibull_min`.

In [ ]:
def ajustar_con_fitter(datos, nombre_var, distribuciones=None, bins=20):
    """
    Ajusta varias distribuciones teóricas a una variable cuantitativa usando fitter.

    Parámetros
    ----------
    datos : array-like
        Vector numérico con los datos.
    nombre_var : str
        Nombre de la variable analizada.
    distribuciones : list[str] | None
        Lista de distribuciones de scipy.stats que se desean comparar.
    bins : int
        Número de intervalos para el histograma.

    Retorna
    -------
    fitter.Fitter
        Objeto Fitter ya ajustado.
    pandas.DataFrame
        Tabla resumen de las mejores distribuciones.
    dict
        Mejor distribución según sumsquare_error.
    """
    if distribuciones is None:
        distribuciones = ["norm", "gamma", "lognorm", "expon", "weibull_min"]

    x = pd.Series(datos).dropna()
    x = x[np.isfinite(x)]

    print(f"Variable analizada: {nombre_var}")
    print(f"n = {len(x)}")
    print(f"Mínimo = {x.min():.4f} | Máximo = {x.max():.4f}")

    f = Fitter(
        x.to_numpy(),
        distributions=distribuciones,
        bins=bins,
        timeout=60,
        verbose=False
    )

    f.fit()
    resumen = f.summary(Nbest=len(distribuciones), plot=True, method="sumsquare_error")
    mejor = f.get_best(method="sumsquare_error")

    print("\nMejor distribución según sumsquare_error:")
    print(mejor)

    return f, resumen, mejor


f_medinc, resumen_medinc, mejor_medinc = ajustar_con_fitter(
    df_housing["MedInc"],
    nombre_var="MedInc"
)

resumen_medinc

---

# 6. Carga de los datasets de la tarea

En la tarea se usarán dos variables principales:

| Área | Dataset | Variable | Justificación |
|---|---|---:|---|
| Ingeniería Agrícola | `Soils` de `carData` | `Conduc` | Conductividad del suelo; variable útil para discutir salinidad y variabilidad físico-química del suelo. |
| Ingeniería Agroindustrial | `acero.csv` | `consumo` | Consumo del proceso; variable útil para modelar variabilidad operativa y planear rangos esperados de operación. |

## 6.1. Dataset `Soils`

En Python se cargará con `statsmodels.datasets.get_rdataset`.

In [ ]:
# ============================================================
# Carga del dataset Soils desde el paquete carData
# ============================================================

soils = get_rdataset("Soils", "carData").data

print(soils.shape)
display(soils.head())

# Variable agrícola seleccionada
x_soils = soils["Conduc"].dropna()

display(generar_tdf(x_soils, nombre_var="Conduc"))

In [ ]:
# ============================================================
# Ajuste de distribuciones para Conduc - Soils
# ============================================================

f_soils, resumen_soils, mejor_soils = ajustar_con_fitter(
    x_soils,
    nombre_var="Soils$Conduc",
    distribuciones=["norm", "gamma", "lognorm", "expon", "weibull_min"],
    bins=10
)

resumen_soils

## 6.2. Dataset `acero.csv`

El archivo `acero.csv` tiene algunas variables con coma decimal.  
La siguiente función convierte automáticamente las columnas tipo texto que representen números con coma decimal.

In [ ]:
def cargar_acero(ruta="acero.csv"):
    """
    Carga el archivo acero.csv y convierte columnas con coma decimal a formato numérico.

    Parámetros
    ----------
    ruta : str
        Ruta del archivo acero.csv.

    Retorna
    -------
    pandas.DataFrame
        DataFrame con las columnas numéricas convertidas.
    """
    acero = pd.read_csv(ruta, sep=",", encoding="utf-8")

    for col in acero.columns:
        if acero[col].dtype == "object":
            texto = acero[col].astype(str).str.replace(",", ".", regex=False)
            convertido = pd.to_numeric(texto, errors="coerce")

            # Convierte la columna si la mayoría de sus valores se reconocen como numéricos.
            proporcion_numerica = convertido.notna().mean()
            if proporcion_numerica >= 0.80:
                acero[col] = convertido

    return acero


# En Google Colab, suba primero el archivo acero.csv al entorno de trabajo.
# from google.colab import files
# files.upload()

acero = cargar_acero("acero.csv")

print(acero.shape)
display(acero.head())
display(acero.dtypes)

# Variable agroindustrial seleccionada
# Explicitly convert 'consumo' to numeric, handling comma decimals, before dropping NA
x_acero = acero["consumo"].astype(str).str.replace(",", ".", regex=False)
x_acero = pd.to_numeric(x_acero, errors="coerce").dropna()

display(generar_tdf(x_acero, nombre_var="consumo"))

In [ ]:
# ============================================================
# Ajuste de distribuciones para consumo - acero.csv
# ============================================================

f_acero, resumen_acero, mejor_acero = ajustar_con_fitter(
    x_acero,
    nombre_var="acero$consumo",
    distribuciones=["norm", "gamma", "lognorm", "expon", "weibull_min"],
    bins=12
)

resumen_acero

---

# 7. Ajuste en R con `fitdistrplus`

El paquete `fitdistrplus` permite ajustar distribuciones paramétricas univariadas, comparar medidas de bondad de ajuste y generar gráficos diagnósticos.

## 7.1. Instalación y carga de paquetes en R

Copie el siguiente bloque en un archivo `.Rmd` en Posit Cloud o RStudio.

```r
# ============================================================
# Instalación de paquetes
# ============================================================

install.packages(c("fitdistrplus", "carData", "dplyr", "ggplot2"))

# ============================================================
# Carga de paquetes
# ============================================================

library(fitdistrplus)
library(carData)
library(dplyr)
library(ggplot2)
```

## 7.2. Función general en R para ajustar distribuciones

Esta función ajusta las distribuciones normal, gamma, lognormal, Weibull y exponencial.  
Debe usarse con variables positivas cuando se incluyan gamma, lognormal, Weibull y exponencial.

```r
ajustar_fitdistrplus <- function(x, nombre_var = "Variable") {

  # Limpieza básica
  x <- na.omit(x)
  x <- x[is.finite(x)]

  cat("Variable analizada:", nombre_var, "\n")
  cat("n =", length(x), "\n")
  cat("Mínimo =", min(x), "| Máximo =", max(x), "\n\n")

  # Exploración preliminar
  hist(x, probability = TRUE,
       main = paste("Histograma de", nombre_var),
       xlab = nombre_var)
  lines(density(x), lwd = 2)

  # Diagrama de Cullen-Frey
  descdist(x, boot = 500)

  # Ajustes paramétricos
  fit_norm <- fitdist(x, "norm")
  fit_gamma <- fitdist(x, "gamma")
  fit_lnorm <- fitdist(x, "lnorm")
  fit_weibull <- fitdist(x, "weibull")
  fit_exp <- fitdist(x, "exp")

  ajustes <- list(
    Normal = fit_norm,
    Gamma = fit_gamma,
    Lognormal = fit_lnorm,
    Weibull = fit_weibull,
    Exponencial = fit_exp
  )

  # Comparación gráfica
  denscomp(ajustes, legendtext = names(ajustes),
           main = paste("Comparación de densidades:", nombre_var))

  cdfcomp(ajustes, legendtext = names(ajustes),
          main = paste("Comparación de FDA:", nombre_var))

  qqcomp(ajustes, legendtext = names(ajustes),
         main = paste("Gráfico Q-Q:", nombre_var))

  ppcomp(ajustes, legendtext = names(ajustes),
         main = paste("Gráfico P-P:", nombre_var))

  # Estadísticos de bondad de ajuste
  bondad <- gofstat(ajustes, fitnames = names(ajustes))
  print(bondad)

  # Tabla comparativa básica
  tabla <- data.frame(
    Distribucion = names(ajustes),
    AIC = bondad$aic,
    BIC = bondad$bic
  )

  print(tabla[order(tabla$AIC), ])

  return(list(
    ajustes = ajustes,
    bondad = bondad,
    tabla = tabla
  ))
}
```

## 7.3. Ajuste en R para `Soils$Conduc`

```r
# ============================================================
# Dataset Soils - Variable Conduc
# ============================================================

data(Soils)

x_soils <- Soils$Conduc

resultado_soils <- ajustar_fitdistrplus(
  x = x_soils,
  nombre_var = "Soils$Conduc"
)
```

## 7.4. Ajuste en R para `acero$consumo`

El archivo `acero.csv` debe estar en el mismo directorio del documento `.Rmd`.

```r
# ============================================================
# Lectura y limpieza de acero.csv
# ============================================================

acero <- read.csv(
  "acero.csv",
  stringsAsFactors = FALSE,
  check.names = FALSE
)

# Conversión de coma decimal a punto decimal
acero$consumo <- as.numeric(gsub(",", ".", acero$consumo))

# Verificación
summary(acero$consumo)

resultado_acero <- ajustar_fitdistrplus(
  x = acero$consumo,
  nombre_var = "acero$consumo"
)
```

---

# 8. Tarea para estudiantes

## Título de la tarea

**Ajuste de datos experimentales a distribuciones de probabilidad teóricas con Python y R**

## Situación problema

En ingeniería es frecuente trabajar con variables continuas afectadas por variabilidad natural, experimental u operacional.  
Antes de usar modelos probabilísticos, intervalos de confianza, simulaciones o criterios de control, es necesario explorar si los datos pueden representarse razonablemente mediante una distribución teórica.

En esta tarea se analizarán dos variables:

1. **`Soils$Conduc`**: conductividad del suelo, de interés en Ingeniería Agrícola.
2. **`acero$consumo`**: consumo del proceso industrial, de interés en Ingeniería Agroindustrial.

---

## Objetivo general

Ajustar datos experimentales de variables continuas a distribuciones de probabilidad teóricas mediante Python y R, comparando resultados gráficos, numéricos y técnicos para seleccionar una distribución razonable en cada caso.

## Objetivos específicos

1. Importar, depurar y describir las variables `Conduc` y `consumo`.
2. Construir tablas de distribución de frecuencias e histogramas con densidad empírica.
3. Ajustar distribuciones teóricas en Python usando `fitter`.
4. Ajustar distribuciones teóricas en R usando `fitdistrplus`.
5. Comparar los resultados obtenidos mediante AIC, BIC, error de ajuste, gráficos Q-Q, P-P, FDA y densidad.
6. Redactar una conclusión técnica sobre la distribución más apropiada para cada variable.

---

## Competencias a desarrollar

- **Competencia estadística:** interpreta la variabilidad de una variable continua mediante frecuencias, densidades y modelos probabilísticos.
- **Competencia computacional:** implementa procedimientos reproducibles en Python y R para ajuste de distribuciones.
- **Competencia analítica:** compara distribuciones candidatas usando criterios gráficos y numéricos.
- **Competencia comunicativa:** redacta conclusiones técnicas claras, justificadas y pertinentes al contexto de ingeniería.
- **Competencia ética en uso de IA:** utiliza chatbots como apoyo para comprender y documentar, sin reemplazar el análisis propio.

---

## Resultados de aprendizaje

Al finalizar la actividad, el estudiante estará en capacidad de:

1. Preparar datos reales o experimentales para análisis probabilístico univariado.
2. Ajustar distribuciones teóricas mediante `fitter` en Python y `fitdistrplus` en R.
3. Interpretar criterios de bondad de ajuste sin depender de un único indicador.
4. Justificar técnicamente la selección de una distribución para una variable de ingeniería.
5. Presentar un informe reproducible con código, resultados, gráficos e interpretación.

---

# 9. Procedimiento paso a paso

## Parte A. Trabajo en Python con `fitter`

1. Crear o abrir un notebook en Google Colab.
2. Instalar y cargar las librerías necesarias.
3. Cargar el dataset `Soils` con `get_rdataset("Soils", "carData")`.
4. Seleccionar la variable `Conduc`.
5. Subir el archivo `acero.csv`.
6. Cargar y limpiar `acero.csv`, convirtiendo la coma decimal a punto decimal.
7. Seleccionar la variable `consumo`.
8. Para cada variable:
   - calcular tamaño muestral, mínimo, máximo, media, mediana, desviación estándar y coeficiente de variación;
   - construir tabla de distribución de frecuencias;
   - graficar histograma con KDE;
   - ajustar las distribuciones `norm`, `gamma`, `lognorm`, `expon` y `weibull_min`;
   - registrar la tabla resumen generada por `fitter`;
   - identificar la mejor distribución según `sumsquare_error`, AIC y BIC;
   - escribir una interpretación técnica.

## Parte B. Trabajo en R con `fitdistrplus`

1. Crear un documento `.Rmd` en Posit Cloud.
2. Instalar y cargar `fitdistrplus` y `carData`.
3. Cargar `Soils` y seleccionar `Conduc`.
4. Cargar `acero.csv` y convertir `consumo` a numérica.
5. Para cada variable:
   - construir histograma y densidad empírica;
   - ejecutar `descdist()` para inspección preliminar;
   - ajustar `norm`, `gamma`, `lnorm`, `weibull` y `exp`;
   - usar `denscomp()`, `cdfcomp()`, `qqcomp()` y `ppcomp()`;
   - comparar con `gofstat()`;
   - organizar una tabla con AIC y BIC;
   - redactar una conclusión comparando los resultados de R con los de Python.

## Parte C. Informe final

El informe debe contener:

1. Portada con nombres de integrantes, programa y grupo.
2. Introducción breve al ajuste de distribuciones.
3. Descripción de los datasets y variables seleccionadas.
4. Procedimiento en Python.
5. Procedimiento en R.
6. Resultados gráficos y tablas comparativas.
7. Discusión técnica por variable.
8. Conclusiones.
9. Evidencia de uso responsable de IA: prompt usado y explicación de cómo fue verificado.
10. Enlaces de entrega: Colab, RMarkdown/HTML y repositorio GitHub si se solicita.

---

# 10. Rúbrica de evaluación

| Criterio | Ponderación | Nivel alto | Nivel medio | Nivel bajo |
|---|---:|---|---|---|
| Preparación y limpieza de datos | 15% | Carga correctamente ambos datasets, convierte variables numéricas y documenta la limpieza. | Carga los datos, pero la limpieza o documentación es parcial. | Presenta errores de carga, conversión o selección de variables. |
| Análisis exploratorio univariado | 15% | Presenta estadísticas, TDF, histogramas y KDE con interpretación clara. | Presenta gráficos o estadísticas, pero con interpretación limitada. | Omite elementos básicos del análisis exploratorio. |
| Ajuste en Python con `fitter` | 20% | Ajusta varias distribuciones, interpreta métricas y justifica la selección. | Ejecuta `fitter`, pero interpreta de forma incompleta. | Solo ejecuta código sin análisis o presenta errores importantes. |
| Ajuste en R con `fitdistrplus` | 20% | Usa `fitdist`, gráficos comparativos y `gofstat` correctamente. | Realiza ajustes básicos, pero con comparación limitada. | No logra ajustar o no interpreta los resultados. |
| Comparación Python-R | 10% | Contrasta coincidencias y diferencias entre herramientas. | Menciona resultados, pero sin contraste técnico suficiente. | No compara los resultados entre Python y R. |
| Conclusión técnica | 10% | Selecciona una distribución por variable con justificación estadística y de ingeniería. | Selecciona distribución, pero con justificación débil. | La conclusión es ausente o no está sustentada. |
| Presentación, reproducibilidad y uso responsable de IA | 10% | Informe ordenado, código reproducible, prompts documentados y verificación propia. | Informe comprensible, pero con fallas de orden o reproducibilidad. | Entrega desordenada, incompleta o dependiente de IA sin verificación. |

## Escala sugerida

- **4.5 a 5.0:** Excelente dominio conceptual, computacional e interpretativo.
- **4.0 a 4.4:** Buen trabajo con detalles menores por mejorar.
- **3.0 a 3.9:** Cumple lo básico, pero requiere fortalecer interpretación y comparación.
- **2.0 a 2.9:** Desarrollo incompleto o con errores metodológicos importantes.
- **0.0 a 1.9:** No evidencia logro de los resultados de aprendizaje.

---

# 11. Formato de evaluación posterior

Cuando el docente adjunte los informes o evidencias de los estudiantes, se puede evaluar cada entrega con la rúbrica anterior.

## Plantilla de registro

| Estudiante / Grupo | Limpieza 15% | EDA 15% | Python 20% | R 20% | Comparación 10% | Conclusión 10% | Presentación 10% | Nota final | Observaciones |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---|

## Criterio de retroalimentación

La retroalimentación debe ser breve, específica y orientada a mejora:

1. Qué hizo bien el grupo.
2. Qué debe corregir.
3. Qué debe fortalecer en interpretación estadística.
4. Qué recomendación se da para futuros análisis de datos experimentales.

---

# 12. Cierre conceptual

El ajuste de distribuciones no consiste únicamente en encontrar la curva que “mejor se vea”.  
Un buen análisis debe integrar:

- forma empírica de los datos;
- naturaleza de la variable;
- soporte de la distribución;
- métricas de ajuste;
- gráficos diagnósticos;
- interpretación técnica del fenómeno.

En ingeniería, el modelo probabilístico debe ser estadísticamente defendible y técnicamente razonable.